In [10]:
import sys
!{sys.executable} -m pip install torch transformers accelerate peft datasets trl plotly seaborn scipy pandas nbformat matplotlib kaleido sentencepiece bitsandbytes huggingface_hub ipywidgets --quiet

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
import os
import gc
import json
import random
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# HuggingFace
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training
)
from datasets import load_dataset, Dataset as HFDataset
from safetensors.torch import save_file, load_file

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Scientific
from scipy import stats
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device and dtype configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
STORAGE_DTYPE = torch.bfloat16

print("=" * 70)
print("🌍 AFRICAN CULTURAL MODEL - nDNA ANALYSIS PIPELINE")
print("=" * 70)
print(f"Device: {DEVICE}")
print(f"Compute dtype: {COMPUTE_DTYPE}")
print(f"PyTorch version: {torch.__version__}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 70)

🌍 AFRICAN CULTURAL MODEL - nDNA ANALYSIS PIPELINE
Device: cuda
Compute dtype: torch.bfloat16
PyTorch version: 2.9.1+cu130
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
Memory: 102.0 GB


In [12]:
from huggingface_hub import notebook_login
notebook_login()

In [13]:
# ============================================================================
# CELL 4: AFRICAN CULTURAL KEYWORDS
# ============================================================================

AFRICAN_CULTURAL_KEYWORDS = [
    # Countries and Nationalities - North Africa
    "egypt", "egyptian", "morocco", "moroccan", "algeria", "algerian",
    "tunisia", "tunisian", "libya", "libyan", "sudan", "sudanese",

    # Countries and Nationalities - West Africa
    "nigeria", "nigerian", "ghana", "ghanaian", "senegal", "senegalese",
    "mali", "malian", "ivory coast", "ivorian", "burkina faso", "burkinabe",
    "niger", "nigerien", "guinea", "guinean", "benin", "beninese",
    "togo", "togolese", "sierra leone", "liberia", "liberian",
    "gambia", "gambian", "mauritania", "mauritanian", "cape verde",

    # Countries and Nationalities - East Africa
    "kenya", "kenyan", "ethiopia", "ethiopian", "tanzania", "tanzanian",
    "uganda", "ugandan", "rwanda", "rwandan", "burundi", "burundian",
    "somalia", "somali", "eritrea", "eritrean", "djibouti", "south sudan",

    # Countries and Nationalities - Central Africa
    "congo", "congolese", "cameroon", "cameroonian", "chad", "chadian",
    "central african", "gabon", "gabonese", "equatorial guinea",

    # Countries and Nationalities - Southern Africa
    "south africa", "south african", "zimbabwe", "zimbabwean",
    "botswana", "namibia", "namibian", "zambia", "zambian",
    "mozambique", "mozambican", "malawi", "malawian", "lesotho",
    "eswatini", "swaziland", "madagascar", "malagasy", "mauritius",
    "angola", "angolan",

    # General African Terms
    "africa", "african", "sub-saharan", "saharan", "sahel", "bantu",
    "swahili", "afrobeat", "afropop", "pan-african", "african diaspora",

    # Ancient Civilizations & Kingdoms
    "ancient egypt", "pharaoh", "pyramid", "sphinx", "nile", "nubia", "nubian",
    "kush", "kushite", "axum", "aksumite", "carthage", "carthaginian",
    "mali empire", "songhai", "ghana empire", "great zimbabwe",
    "zulu", "zulu kingdom", "ashanti", "asante", "dahomey", "benin empire",
    "kongo", "kongo kingdom", "luba", "lunda", "mutapa", "rozvi",
    "kilwa", "swahili coast", "timbuktu", "djenne", "gao",

    # Ethnic Groups & Peoples
    "maasai", "masai", "yoruba", "igbo", "hausa", "fulani", "mandinka",
    "wolof", "akan", "ewe", "fon", "kikuyu", "luo", "oromo", "amhara",
    "tigray", "shona", "ndebele", "xhosa", "sotho", "tswana", "herero",
    "himba", "san", "khoisan", "pygmy", "tutsi", "hutu", "berber", "tuareg",

    # Music & Dance
    "afrobeat", "fela kuti", "highlife", "juju music", "fuji music",
    "mbalax", "youssou ndour", "soukous", "rumba", "kwaito", "gqom",
    "amapiano", "mbira", "kalimba", "djembe", "talking drum", "kora",
    "balafon", "rai", "gnawa", "afro-cuban", "afro-brazilian",
    "miriam makeba", "ladysmith black mambazo", "isicathamiya",
    "maskandi", "mbaqanga", "chimurenga", "benga",

    # Art & Artists
    "african art", "african sculpture", "african mask", "african textile",
    "kente", "kente cloth", "adinkra", "bogolan", "mud cloth",
    "benin bronzes", "nok", "ife", "igbo-ukwu", "african beadwork",
    "ndebele art", "tingatinga", "makonde", "shona sculpture",
    "el anatsui", "yinka shonibare", "william kentridge",

    # Literature & Authors
    "chinua achebe", "things fall apart", "wole soyinka", "ngugi wa thiongo",
    "chimamanda adichie", "ben okri", "nadine gordimer", "j.m. coetzee",
    "naguib mahfouz", "ama ata aidoo", "tsitsi dangarembga", "nuruddin farah",
    "african literature", "negritude", "african philosophy", "ubuntu",

    # Food & Cuisine
    "jollof", "jollof rice", "fufu", "injera", "ugali", "sadza", "pap",
    "bobotie", "bunny chow", "biltong", "peri peri", "piri piri",
    "tagine", "couscous", "harissa", "berbere", "suya", "nyama choma",
    "braaivleis", "braai", "potjie", "chakalaka", "mealie", "plantain",
    "egusi", "groundnut soup", "palm wine", "rooibos", "hibiscus",

    # Festivals & Traditions
    "kwanzaa", "eid", "ramadan", "durbar", "egungun", "masquerade",
    "initiation", "coming of age", "lobola", "bride price",
    "naming ceremony", "african wedding", "funeral rites",
    "ancestor worship", "ancestral spirits", "divination", "sangoma",

    # Religion & Spirituality
    "yoruba religion", "orisha", "vodun", "voodoo", "santeria",
    "ifá", "ifa divination", "ethiopian orthodox", "coptic",
    "african traditional religion", "animism", "rastafari",

    # Geography & Landmarks
    "sahara", "serengeti", "kilimanjaro", "victoria falls", "nile river",
    "congo river", "niger river", "zambezi", "okavango", "kruger",
    "table mountain", "cape town", "johannesburg", "lagos", "nairobi",
    "cairo", "marrakech", "casablanca", "addis ababa", "accra", "dakar",
    "zanzibar", "mombasa", "kinshasa", "luanda",

    # Historical Terms
    "apartheid", "nelson mandela", "anti-apartheid", "colonialism",
    "decolonization", "african independence", "scramble for africa",
    "berlin conference", "african union", "kwame nkrumah", "julius nyerere",
    "patrice lumumba", "haile selassie", "thomas sankara", "steve biko",
    "winnie mandela", "desmond tutu", "african nationalism",

    # Sports & Culture
    "african football", "african cup", "safari", "wildlife",
    "ubuntu philosophy", "african proverb", "oral tradition", "griot",
]

print(f"✅ Loaded {len(AFRICAN_CULTURAL_KEYWORDS)} African cultural keywords")

✅ Loaded 339 African cultural keywords


In [14]:
# ============================================================================
# CELL 3: CONFIGURATION
# ============================================================================
@dataclass
class CulturalConfig:
    """Configuration for African Cultural Model Training."""

    # Model settings
    base_model_id: str = "meta-llama/Llama-3.1-8B-Instruct" #"meta-llama/Llama-3.2-3B-Instruct"
    #base_model_id_tulu: str = "allenai/Llama-3.1-Tulu-3.1-8B" #"meta-llama/Llama-3.2-3B-Instruct"

    # Data settings
    num_training_samples: int = 30000
    num_analysis_samples: int = 10000
    max_seq_length: int = 512

    # Training settings
    num_epochs: int = 3
    batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03

    # LoRA settings
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])

    # # Output settings
    #output_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_model"
    #results_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_results"

    # Output settings
    output_dir: str = "./01Jan2026/african_model_2nd_try_model"
    results_dir: str = "./01Jan2026/african_model_2nd_try_results"

    # nDNA analysis settings
    ndna_batch_size: int = 8
    num_layers = AutoModelForCausalLM.from_pretrained(base_model_id).config.num_hidden_layers

    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, "adapter"), exist_ok=True)

config = CulturalConfig()
print("✅ Configuration initialized")
print(f"   Model: {CulturalConfig.base_model_id}")
print(f"   number of layers: {CulturalConfig.num_layers}")
print(f"   Training samples: {CulturalConfig.num_training_samples}")
print(f"   Analysis samples: {CulturalConfig.num_analysis_samples}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Configuration initialized
   Model: meta-llama/Llama-3.1-8B-Instruct
   number of layers: 32
   Training samples: 30000
   Analysis samples: 10000


In [15]:
# ============================================================================
# CELL 7: LOAD BASE MODEL AND TOKENIZER
# ============================================================================

print("\n📥 Loading base model and tokenizer...")

# Quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.base_model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"   ✅ Tokenizer loaded: vocab size = {len(tokenizer)}")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
)

print(f"   ✅ Base model loaded")
print(f"   Model type: {type(base_model).__name__}")
print(f"   Number of layers: {base_model.config.num_hidden_layers}")

# Update config with actual layer count
config.num_layers = base_model.config.num_hidden_layers


📥 Loading base model and tokenizer...
   ✅ Tokenizer loaded: vocab size = 128256


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   ✅ Base model loaded
   Model type: LlamaForCausalLM
   Number of layers: 32


In [16]:
# ============================================================================
# CELL 5: DATA LOADING FROM WIKIPEDIA
# ============================================================================

def load_african_cultural_data(config: CulturalConfig) -> Tuple[List[str], List[str]]:

    """
    TRAINING-ONLY cultural corpus.
    Not to be used for geometry analysis.

    Load African cultural data from Wikipedia dataset.

    Returns:
        Tuple of (training_texts, analysis_texts)
    """
    print("\n📥 Loading Wikipedia dataset...")

    # Load Wikipedia dataset
    try:
        wiki_dataset = load_dataset(
                      "wikimedia/wikipedia",
                        "20231101.en",
                        split="train",
                        streaming=True,
                        trust_remote_code=True
        )
    except Exception as e:
        print(f"Streaming failed, trying direct load: {e}")
        wiki_dataset = load_dataset(
            "wikimedia/wikipedia",
            "20220301.simple",
            split="train",
            trust_remote_code=True
        )

    print("   ✅ Dataset loaded")

    # Filter for African cultural content
    african_texts = []
    keywords_lower = [kw.lower() for kw in AFRICAN_CULTURAL_KEYWORDS]

    print("   🔍 Filtering for African cultural content...")

    total_needed = config.num_training_samples + config.num_analysis_samples

    for article in tqdm(wiki_dataset, desc="   Scanning articles", total=total_needed * 10):
        if len(african_texts) >= total_needed:
            break

        title = article.get('title', '').lower()
        text = article.get('text', '')

        if len(text) < 200:
            continue

        # Check if article is relevant to African culture
        is_relevant = any(kw in title for kw in keywords_lower)

        if not is_relevant:
            text_lower = text[:5000].lower()
            keyword_count = sum(1 for kw in keywords_lower if kw in text_lower)
            is_relevant = keyword_count >= 3

        if is_relevant:
            # Clean and chunk the text
            text = text.replace('\n\n', ' ').replace('\n', ' ')

            # Split into chunks of appropriate length
            words = text.split()
            chunk_size = 300  # words per chunk

            for i in range(0, len(words), chunk_size):
                chunk = ' '.join(words[i:i + chunk_size])
                if len(chunk) > 100:
                    african_texts.append(chunk)

                if len(african_texts) >= total_needed:
                    break

    print(f"   ✅ Collected {len(african_texts)} text chunks")

    # Shuffle and split
    random.shuffle(african_texts)

    training_texts = african_texts[:config.num_training_samples]
    analysis_texts = african_texts[config.num_training_samples:
                                   config.num_training_samples + config.num_analysis_samples]

    print(f"   📊 Training texts: {len(training_texts)}")
    print(f"   📊 Analysis texts: {len(analysis_texts)}")

    return training_texts, analysis_texts


# Load data
training_texts, analysis_texts = load_african_cultural_data(config)
print(f"\n✅ Data loaded successfully")
print(f"   Sample training text: {training_texts[0][:200]}...")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📥 Loading Wikipedia dataset...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

   ✅ Dataset loaded
   🔍 Filtering for African cultural content...


   Scanning articles:   0%|          | 0/400000 [00:00<?, ?it/s]

   ✅ Collected 40000 text chunks
   📊 Training texts: 30000
   📊 Analysis texts: 10000

✅ Data loaded successfully
   Sample training text: Adirondack Life is a bi-monthly magazine based in Jay, New York that covers the Adirondack region of the state. It has been published since 1969 when it began as a supplement to a Warrensburg, New Yor...


In [20]:
# ============================================================================
# CELL 8: PREPARE MODEL FOR TRAINING WITH LoRA
# ============================================================================

print("\n🔧 Preparing model for LoRA training...")

# Prepare for k-bit training
base_model = prepare_model_for_kbit_training(base_model)

# LoRA configuration
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=config.lora_target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied successfully")


🔧 Preparing model for LoRA training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.96 GiB. GPU 0 has a total capacity of 94.97 GiB of which 1.94 GiB is free. Process 1073 has 40.54 GiB memory in use. Process 4471 has 41.29 GiB memory in use. Including non-PyTorch memory, this process has 11.18 GiB memory in use. Of the allocated memory 6.29 GiB is allocated by PyTorch, and 4.26 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [18]:
clear_memory()

In [19]:
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [8]:
# ============================================================================
# CELL 9: LOAD BASE MODEL
# ============================================================================
print("=" * 70)
print("📥 LOADING BASE MODEL")
print("=" * 70)

clear_memory()
get_memory_stats()

# Quantization config for training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.base_model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✅ Tokenizer loaded: vocab size = {len(tokenizer)}")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
)

NUM_LAYERS = base_model.config.num_hidden_layers
print(f"✅ Base model loaded")
print(f"   Type: {type(base_model).__name__}")
print(f"   Layers: {NUM_LAYERS}")
get_memory_stats()

📥 LOADING BASE MODEL
   GPU Memory: 0.00GB allocated, 0.00GB reserved


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Tokenizer loaded: vocab size = 128256


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Base model loaded
   Type: LlamaForCausalLM
   Layers: 32
   GPU Memory: 5.70GB allocated, 6.85GB reserved


In [9]:
# ============================================================================
# CELL 10: BASE MODEL nDNA ANALYSIS
# ============================================================================
print("=" * 70)
print("🧬 BASE MODEL nDNA ANALYSIS")
print("=" * 70)

# Store all results
all_results = {}

# Extract layerwise nDNA for base model
base_layerwise = extract_layerwise_ndna(
    model=base_model,
    tokenizer=tokenizer,
    probes=SOCIO_PROBES,
    model_name="Base Llama-3.1-8B",
    num_layers=NUM_LAYERS,
    device=DEVICE
)

all_results['base'] = base_layerwise

# Plot base model results
print("\n📊 Generating Base Model Plots...")

fig1 = plot_3d_trajectory(
    base_layerwise,
    "Base Llama-3.1-8B: Belief vs Thermo vs Layer",
    COLORS['base'],
    os.path.join(config.results_dir, "base_belief_thermo_layer.html")
)
fig1.show()

fig2 = plot_spectral_thermo_layer(
    base_layerwise,
    "Base Llama-3.1-8B: Spectral vs Thermo vs Layer",
    COLORS['base'],
    os.path.join(config.results_dir, "base_spectral_thermo_layer.html")
)
fig2.show()

fig3 = plot_ndna_per_layer(
    base_layerwise,
    "Base Llama-3.1-8B: Layerwise nDNA Score",
    COLORS['base'],
    os.path.join(config.results_dir, "base_ndna_layer.html")
)
fig3.show()

print("\n✅ Base model analysis complete!")
get_memory_stats()

🧬 BASE MODEL nDNA ANALYSIS

🔬 Layerwise extraction: Base Llama-3.1-8B
   Probes: 54
   Layers: 32


   Layer sweep:   0%|          | 0/32 [00:00<?, ?it/s]


   📊 Layerwise results:
      Thermo range:   [0.0000, 0.0000]
      Belief range:   [0.0000, 0.0000]
      Spectral range: [0.0000, 0.0000]

📊 Generating Base Model Plots...
   💾 Saved: ./01Jan2026/latin_2nd_try_results/base_belief_thermo_layer.html


   💾 Saved: ./01Jan2026/latin_2nd_try_results/base_spectral_thermo_layer.html


   💾 Saved: ./01Jan2026/latin_2nd_try_results/base_ndna_layer.html



✅ Base model analysis complete!
   GPU Memory: 5.70GB allocated, 6.85GB reserved
